# Everytown dbt demo & discussion

This is a pointed discussion, demo, and Q&A session about using dbt in Civis Platform.

## Topics of Discussion

1. Setting up a dbt projects and profile configuration, including any templates or best practices for faster set ups
2. A suggested workflow for developing and productionalizing dbt, including when to use Civis Studio versus the dbt runner in a job or workflow
3. How retries work in Civis, including what happens to ephemeral models when a job fails and how to handle your hourly table build
4. ???

## 1. Project Setup

### a) Review profiles.yml. 
- Required for development in Civis Studio. Can be autogenerated when using a dbt platform job
- Use [our public profiles.yml](https://github.com/civisanalytics/platform-code-examples/blob/efe1a71e81da72d6093e18bced2effdb76cc0e8e/dbt/profiles.yml) to get started.

### b) Set up Civis Studio Parameters accordingly
- Dictated by profiles.yml
- Show parameters based on Civis's publicly available profiles.yml

In [1]:
!cat /workspace/platform-code-examples/dbt/profiles.yml

dbt-civis:
  target: redshift_adapter
  outputs:
    redshift_adapter:
      type: "{{ env_var('DATABASE_TYPE') }}"
      host: "{{ env_var('DATABASE_HOST') }}"
      port: "{{ env_var('DATABASE_PORT') | as_number }}"
      user: "{{ env_var('DATABASE_USERNAME') }}"
      pass: "{{ env_var('DBT_ENV_SECRET_DATABASE_PASSWORD') }}"
      dbname: "{{ env_var('DATABASE_NAME') }}"
      schema: "{{ env_var('DBT_SCHEMA') }}"
      threads: 4

    bigquery_service_account_adapter:
      type: "{{ env_var('DATABASE_TYPE') }}"
      method: "{{ env_var('GCP_AUTH_METHOD') }}"
      project: "{{ env_var('GCP_PROJECT_ID') }}"
      schema: "{{ env_var('DBT_SCHEMA') }}"
      keyfile: "{{ env_var('GCP_SERVICE_ACCOUNT_KEYFILE_JSON') }}"
      threads: 4

    bigquery_oauth_adapter:
      threads: 4 # Must be a value of 1 or greater
      type: "{{ env_var('DATABASE_TYPE') }}"
      method: "{{ env_var('GCP_AUTH_METHOD') }}"
      project: "{{ env_var('GCP_PROJECT_ID') }}"
      schema: "{{ env_var('D

### c) Execute dbt debug

1. CLI - Use a bash terminal to run commands. This offers the most flexibility.
2. dbt extension - Mostly nice for additional context on a dbt project. It can execute build/run/test commands, but not debug/retry

In [1]:
%%sh
cd /workspace/platform-code-examples/dbt
dbt debug

       dbt 2.0.9
   Loading profiles.yml
 Debugging profile: redshift_adapter
 Debugging dbt version: 2.0.9
 Debugging platform: linux x86_64 (unix)
 Debugging adapter type: redshift (remote)
 Debugging dependencies:
  git: OK
 Debugging connection:
  "port": 5432,
  "database": "dev",
  "schema": "jaffle_demo",
  "host": "redshift.acme.civis.io",
  "user": "jsmith"
 Debugging connection test: OK
  Debugged All checks passed!

==================== Execution Summary =====================
Finished 'debug' successfully for target 'redshift_adapter' s]


## 2. Building a project for productionalization

### General Tips

- Use one big dbt project instead of multiple smaller ones to take advantage of code reuse.
- Create a folder structure that makes sense for your domain. A good default option is stg/intermediate/marts to cover E/T/L
- Use tags to define and execute pipelines. 
- Use inheritence in `dbt_project.yml` to cut down on redundant configuration (show /workspace/platform-code-examples/dbt/dbt_project.yml)
- Use vars and macros to abstract functionality as much as is reasonable.

In [7]:
%%sh
cd /workspace/platform-code-examples/dbt/
# Build everything
# dbt build --full-refresh --vars "{'load_source_data': true}"

# Build just the customer graph
dbt build --select +customers


       dbt 2.0.9
   Loading profiles.yml
   Loading packages.yml
 Succeeded model jaffle_demo.stg_products (view) [3 of 35 in 4.52s]
 Succeeded model jaffle_demo.stg_customers (view) [1 of 35 in 4.74s]
 Succeeded model jaffle_demo.stg_orders (view) [4 of 35 in 4.74s]
 Succeeded model jaffle_demo.stg_supplies (view) [2 of 35 in 4.74s]
    Passed test  unique_stg_products_product_id [7 of 35 in 0.49s]
    Passed test  not_null_stg_products_product_id [6 of 35 in 0.51s]
    Passed test  not_null_stg_customers_customer_id [8 of 35 in 0.33s]
    Passed test  unique_stg_customers_customer_id [9 of 35 in 0.41s]
    Passed test  dbt_utils_expression_is_true_stg_orders_order_total_tax_paid_subtotal [11 of 35 in 0.67s]
    Passed test  not_null_stg_orders_order_id [12 of 35 in 0.68s]
    Passed test  unique_stg_orders_order_id [10 of 35 in 0.69s]
    Passed test  not_null_stg_supplies_supply_uuid [14 of 35 in 0.80s]
    Passed test  unique_stg_supplies_supply_uuid [13 of 35 in 0.83s]
 Succeeded 


=================== Errors and Warnings ====================
[warning] [InvalidConfig (dbt1005)]: Skipping semantic manifest validation due to: No dbt_cloud.yml config. Set DBT_ENGINE_NO_WARN_SEMANTIC_MANIFEST_VALIDATION to suppress this warning.



==================== Execution Summary =====================
Finished 'build' with 1 warning for target 'redshift_adapter' [22.3s]
Processed: 8 models | 25 tests | 2 unit tests
Summary: 35 total | 35 success


### Metricflow

Metricflow is a tool provided by dbt that layers semantic layer and metric definitions into a data warehouse defined in dbt. 

Semantic layers mix inject business context into the data structure that make it easier to answer questions across tables for less technical users.

When a semantic layer exists, **metrics** can be defined which act as aliases for repeatable queries. These queries can be saved using saved queries, and then exported to tables.

The combination of a semantic layer and KPIs enable data governance practices, which are table stakes for BI using AI.

*These commands are currently only available when running in Civis Studio*

In [ ]:
%%sh

# Install dbt-metricflow so we can compute KPIs via the CLI
pip install dbt-metricflow

In [2]:
%%sh
cd /workspace/platform-code-examples/dbt
mf validate-configs
mf list metrics

(To see warnings and future-errors, run again with flag `--show-all`)
✔ 🎉 Successfully parsed manifest from dbt project
✔ 🎉 Successfully validated the semantics of built manifest (ERRORS: 0, FUTURE_ERRORS: 0, WARNINGS: 0)
✔ 🎉 Successfully validated semantic models against data warehouse (ERRORS: 0, FUTURE_ERRORS: 0,warehouse...⠋ Validating semantic models against data warehouse... Validating semantic models against data warehouse...⠹ Validating semantic models against data warehouse... Validating semantic models against data warehouse... Validating semantic models against data warehouse... Validating semantic models against data warehouse... WARNINGS: 0)
✔ 🎉 Successfully validated dimensions against data warehouse (ERRORS: 0, FUTURE_ERRORS: 0, WARehouse...⠇ Validating dimensions against data warehouse...⠙ Validating dimensions against data warehouse...⠸ Validating dimensions against data warehouse...⠼ Validating dimensions against data warehouse... Validating dimensions against data wa

In [5]:
%%sh
cd /workspace/platform-code-examples/dbt

# Compute KPIs
# Defined in /workspace/platform-code-examples/dbt/models/marts/customers.yml#L57
mf query --metrics count_lifetime_orders --group-by metric_time__month

mf query --metrics average_order_value --group-by customer__customer_name

✔ Success 🦄 - query completed after 1.46 seconds
metric_time__month      count_lifetime_orders
--------------------  -----------------------
2024-09-01T00:00:00                     19977
2024-11-01T00:00:00                      5643
2025-03-01T00:00:00                     17497
2025-08-01T00:00:00                       383
2024-10-01T00:00:00                      1518
2025-01-01T00:00:00                      2990
2025-02-01T00:00:00                      1534
2025-05-01T00:00:00                      3656
2025-06-01T00:00:00                      2382
2024-12-01T00:00:00                      4270
2025-04-01T00:00:00                       974
2025-07-01T00:00:00                      1124
✔ Success 🦄 - query completed after 1.50 seconds query…
customer__customer_name      average_order_value
-------------------------  ---------------------
Aaron Gardner                           87.6774
Tracy Garcia                            11.3605
Dylan Anderson                          11.6562
Derek Smi

In [ ]:
%%sh
cd /workspace/platform-code-examples/dbt
# Multiple metrics in one view
mf query --saved-query customer_order_metrics


✔ Initiating query… Initiating query…⠼ Initiating query… Success 🦄 - query completed after 1.72 seconds
customer                                count_lifetime_orders    lifetime_spend_pretax    average_order_value
------------------------------------  -----------------------  -----------------------  ---------------------
c6fb82d9-b6e6-445a-a3d3-0158405a9d16                      194                     1084                5.58763
d31efffd-fdc4-4168-ac26-83b67a6c5c47                      213                     1220                5.7277
fe2e9f59-c77b-4756-ad2e-fe864a560344                      146                     1445                9.89726
34a9dd73-95d0-45d9-a3fb-636e4e080a6c                      187                     1027                5.49198
cbda2de6-06db-4a9f-91af-cc36e8ba0bad                       86                      964               11.2093
5b2179c5-dd1c-45b4-b99c-10f1c7c201f4                      128                     1277                9.97656
a792d50a-1570-450f

## Civis Studio & Platform dbt jobs

**Civis Studio** is a fully fledged IDE hosted in Civis Platform. It contains a wealth of tools to develop and troubleshoot data pipelines. It is most useful for quick iteration when developing pipelines.

**Platform dbt jobs** provide a convenient mechanism for invoking dbt pipelines. They have the residual benefit being a platform object and interopt with Platform Workflow nicely. They are best used for the productionalization of data pipelines.

### New Data Mart Example

Let's say that I want to build a new `geo` data mart, which contains information geographical and census information about the markets my Jaffle shop operates in. I might tackle this in these steps.

Prerequisites
- A schema for development tables is created or set aside.

1. Use **Platform Data Import** to bring the geo data into redshift.
2. Use **Claude Code/Copilot** and the **Civis MCP Connector** inside **Civis Studio** to look up table information about the table I just imported and draft an implementation plan.
3. Review implementaton plan and validate that it achieves the desired goals. In this case, I would want to consider how this information drives existing KPIs. I'd also want to make sure my assumptions about the data are codified in tests.
4. Agent to implement
5. Execute `dbt build` in the Terminal tab in **Civis Studio**. Iterate until successful
6. Stage, commit, and push to Github via CLI or the built-in VSCode extension.
7. Execute the **dbt Platform job** or **Platform Workflow** that the dbt pipeline currently executes in.
8. Validate that the data are correct.

### New Data Pipeline

Let's say I don't have a dbt pipeline at all and I want to set up a production-ready pipeline from scratch. Let's assume that I have already done requirements gathering, EDA, and so I have a good sense of what this pipeline needs to accomplish. I might follow these steps.

1. Create a Platform Workflow based on an existing workflow, or via an AI tool using the **Civis MCP Server** 2. Add **Platform Data Import** to import the datasets into Civis Platform.
3. Add a placeholder dbt task, and maybe a placeholder report task if necessary.
4. Open **Civis Studio**.
5. Create a folder to house data pipelines in your codebase, e.g., `dbt`, `pipelines`
6. Instatiate a new dbt project using `dbt init` using the terminal
7. Copy [profiles.yml](https://github.com/civisanalytics/platform-code-examples/blob/efe1a71e81da72d6093e18bced2effdb76cc0e8e/dbt/profiles.yml) into your dbt folder. Change the target names to taste.
8. Create dbt models mart-by-mart according to spec by hand or with your favorite AI coding tool.
9. Test on **Civis Studio**
10. Update the placeholder dbt task in the workflow to use the command you executed in **Civis Studio**
11. Stage, Commit, and push to Github.
12. Execute the Platform Workflow, which is executing a **dbt Platform Job**


